In [ ]:
!apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 \
    libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 \
    libgbm1 libasound2 libpango-1.0-0 libpangocairo-1.0-0 \
    tesseract-ocr tesseract-ocr-spa -q
!pip install playwright pytesseract pillow pandas openpyxl -q
!python -m playwright install chromium

In [ ]:
from playwright.async_api import async_playwright
import pytesseract
print("✓ Todo instalado correctamente")

In [ ]:
# ── Sección 3: Selector de Región ─────────────────────────────────────────────
REGIONES = [
    "Perú",             # 1
    "Amazonas",         # 2
    "Áncash",           # 3
    "Apurímac",         # 4
    "Arequipa",         # 5
    "Ayacucho",         # 6
    "Cajamarca",        # 7
    "Callao",           # 8
    "Cusco",            # 9
    "Huancavelica",     # 10
    "Huánuco",          # 11
    "Ica",              # 12
    "Junín",            # 13
    "La Libertad",      # 14
    "Lambayeque",       # 15
    "Lima Metropolitana", # 16
    "Lima Provincias",  # 17
    "Loreto",           # 18
    "Madre de Dios",    # 19
    "Moquegua",         # 20
    "Pasco",            # 21
    "Piura",            # 22
    "Puno",             # 23
    "San Martín",       # 24
    "Tacna",            # 25
    "Tumbes",           # 26
    "Ucayali",          # 27
]

print("╔══════════════════════════════════╗")
print("║      SELECTOR DE REGIÓN          ║")
print("╠══════════════════════════════════╣")
for i, r in enumerate(REGIONES, 1):
    print(f"║  {i:>2}. {r:<28}║")
print("╚══════════════════════════════════╝")

print("\nOpciones de selección:")
print("  • Un número        → ej: 18")
print("  • Varios números   → ej: 18,24,27")
print("  • Rango            → ej: 3-10")
print("  • Todo             → ej: all\n")

entrada = input("Seleccioná región/es: 27").strip().lower()

def parsear_seleccion(entrada, regiones):
    indices = set()
    if entrada == "all":
        return list(regiones)
    for parte in entrada.split(","):
        parte = parte.strip()
        if "-" in parte:
            inicio, fin = parte.split("-")
            indices.update(range(int(inicio), int(fin) + 1))
        elif parte.isdigit():
            indices.add(int(parte))
    seleccionadas = []
    for i in sorted(indices):
        if 1 <= i <= len(regiones):
            seleccionadas.append(regiones[i - 1])
        else:
            print(f"  ⚠ Número fuera de rango ignorado: {i}")
    return seleccionadas

REGIONES_SELECCIONADAS = parsear_seleccion(entrada, REGIONES)

if not REGIONES_SELECCIONADAS:
    raise ValueError("❌ No se seleccionó ninguna región válida.")

print(f"\n✓ Regiones seleccionadas ({len(REGIONES_SELECCIONADAS)}):")
for r in REGIONES_SELECCIONADAS:
    print(f"   → {r}")

In [ ]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
from google.colab import files

EMBED_URL = "https://app.powerbi.com/view?r=eyJrIjoiNTExZWMwZDEtMDIzMi00ZGZmLThmOWItZTBjNGU0NDI5ZmRkIiwidCI6IjY0ZWIzYjM3LTE4YmEtNDE2ZC05NDE3LTg3NjMzMmM4YTRkMSIsImMiOjR9"

MES_MAP = {
    "Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Ago": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dic": 12,
    "Jan": 1, "Apr": 4, "Aug": 8, "Dec": 12
}

ACTIVIDADES = ["Agrícola", "Minería", "Manufactura", "Construcción", "Comercio", "Servicios", "No precisa"]

# ── Helpers ────────────────────────────────────────────────────────────────────
async def scroll_panel(page, amount):
    await page.mouse.move(155, 500)
    await page.mouse.wheel(delta_x=0, delta_y=amount)

async def reset_panel_scroll(page):
    """Lleva el panel izquierdo al tope superior."""
    for _ in range(20):
        await page.mouse.move(155, 400)
        await page.mouse.wheel(delta_x=0, delta_y=-500)
        await asyncio.sleep(0.05)
    await asyncio.sleep(1.5)

async def seleccionar_region(page, region):
    """
    Selecciona la región usando el checkbox del panel izquierdo.
    Evita coincidir con texto en headers, logos o gráficos (ej: 'Perú').
    """
    await reset_panel_scroll(page)

    # Intento 1: selectores semánticos de Power BI
    selectores = [
        f'[role="option"]:has-text("{region}")',
        f'[role="listitem"]:has-text("{region}")',
        f'[role="checkbox"][aria-label="{region}"]',
    ]
    for selector in selectores:
        try:
            locs = page.locator(selector)
            count = await locs.count()
            for i in range(count):
                loc = locs.nth(i)
                box = await loc.bounding_box()
                if not box:
                    continue
                print(f"    [debug intento1] selector='{selector}' #{i} → x={box['x']:.0f}, y={box['y']:.0f}")
                if box["x"] > 460:  # ← descartar elementos fuera del panel
                    continue
                await loc.scroll_into_view_if_needed()
                await asyncio.sleep(0.5)
                await loc.click()
                await asyncio.sleep(3)
                print(f"    ✓ Región seleccionada: {region}")
                return
        except:
            continue

    # Intento 2: bounding_box restringido al panel izquierdo (x < 460)
    print(f"    [!] Usando fallback por bounding_box para: {region}")
    for _ in range(30):
        try:
            locs = page.locator(f"text={region}")
            count = await locs.count()
            for i in range(count):
                loc = locs.nth(i)
                box = await loc.bounding_box()
                if box:
                    print(f"    [debug ALL] '{region}' #{i} → x={box['x']:.0f}, y={box['y']:.0f}")
                if box and box["x"] < 460:
                    # ── DIAGNÓSTICO: imprimí TODOS los elementos encontrados ──
                    print(f"    [debug] '{region}' elemento #{i} → x={box['x']:.0f}, y={box['y']:.0f}, w={box['width']:.0f}, h={box['height']:.0f}")
                    await loc.scroll_into_view_if_needed()
                    await asyncio.sleep(0.3)
                    await loc.click()
                    await asyncio.sleep(3)
                    print(f"    ✓ Región seleccionada (fallback): {region}")
                    return
        except:
            pass
        await page.mouse.move(155, 400)
        await page.mouse.wheel(delta_x=0, delta_y=200)
        await asyncio.sleep(0.2)

    raise Exception(f"No se pudo seleccionar la región: {region}")

async def scroll_y_extraer(page):
    tabla_x, tabla_y = 700, 600
    todas_las_filas = {}

    for _ in range(10):
        await page.mouse.move(tabla_x, tabla_y)
        await page.mouse.wheel(delta_x=0, delta_y=-500)
        await asyncio.sleep(0.1)
    await asyncio.sleep(1)

    sin_nuevas = 0
    while sin_nuevas < 5:
        datos = await page.evaluate("""() => {
            const grid = document.querySelector('[role="grid"]');
            if (!grid) return [];
            const rows = grid.querySelectorAll('[role="row"]');
            const result = [];
            for (const row of rows) {
                const cells = [...row.querySelectorAll(
                    '[role="gridcell"], [role="rowheader"], [role="columnheader"]'
                )].map(c => c.innerText.trim());
                if (cells.length >= 2) result.push(cells);
            }
            return result;
        }""")

        nuevas = 0
        for fila in datos:
            if len(fila) >= 2:
                key = fila[0]
                if key and key != 'V_AÑO_MES (grupos), V_AÑO_MES' and key not in todas_las_filas:
                    todas_las_filas[key] = fila[1]
                    nuevas += 1

        if nuevas == 0:
            sin_nuevas += 1
        else:
            sin_nuevas = 0

        await page.mouse.move(tabla_x, tabla_y)
        await page.mouse.wheel(delta_x=0, delta_y=150)
        await asyncio.sleep(0.4)

    print(f"    ✓ {len(todas_las_filas)} filas extraídas")
    return todas_las_filas

# ── Procesador por combinación región + actividad ─────────────────────────────
async def procesar_combinacion(page, region, actividad):
    # ── Reset al inicio ────────────────────────────────────────────────────────
    try:
        await page.locator("text=INICIO").first.click()
        await asyncio.sleep(3)
        await page.locator("text=EVOLUCIÓN").first.click()
        await asyncio.sleep(6)
    except:
        pass

    # ── Seleccionar región (con reset de scroll y selector seguro) ─────────────
    await seleccionar_region(page, region)

    # ── Resetear scroll al tope ANTES del slider ───────────────────────────────
    # Crítico para regiones al final de la lista (Ucayali, Tumbes, etc.)
    await reset_panel_scroll(page)

    # ── Slider todos los años ──────────────────────────────────────────────────
    await page.mouse.click(787, 338)
    await asyncio.sleep(3)

    # ── Bajar hasta ACTIVIDAD ECONÓMICA ───────────────────────────────────────
    for _ in range(15):
        await scroll_panel(page, 300)
        await asyncio.sleep(0.15)
    await asyncio.sleep(2)

    # ── Seleccionar actividad ──────────────────────────────────────────────────
    await page.locator(f"text={actividad}").first.click()
    await asyncio.sleep(3)

    # ── Subir al tope del panel ────────────────────────────────────────────────
    for _ in range(20):
        await scroll_panel(page, -300)
        await asyncio.sleep(0.1)
    await asyncio.sleep(2)

    # ── Click derecho → Mostrar como tabla ────────────────────────────────────
    await page.mouse.click(835, 470, button="right")
    await asyncio.sleep(2)
    try:
        opcion = page.locator("text=Mostrar como tabla").first
        if await opcion.count() == 0:
            opcion = page.locator("text=Show as a table").first
        await opcion.wait_for(timeout=5000)
        await opcion.click()
        await asyncio.sleep(5)
    except Exception as e:
        print(f"    [!] Mostrar tabla: {e}")
        return []

    filas_dict = await scroll_y_extraer(page)

    # ── Volver al informe ──────────────────────────────────────────────────────
    for texto in ["Back to report", "Volver al informe"]:
        try:
            await page.locator(f"text={texto}").first.click()
            await asyncio.sleep(3)
            break
        except:
            pass

    # ── Parsear registros ──────────────────────────────────────────────────────
    registros = []
    for fecha_raw, valor_raw in filas_dict.items():
        try:
            partes = fecha_raw.split(',')
            anio = int(partes[0].strip())
            mes_str = partes[1].strip().split('-')[0].strip()
            mes_num = MES_MAP.get(mes_str)
            if mes_num is None:
                print(f"    [!] Mes no reconocido: '{mes_str}'")
                continue
            empleo = int(valor_raw.replace(',', '').replace('.', '').replace(' ', ''))
            registros.append({
                'AÑO':                 anio,
                'MES':                 mes_num,
                'REGIÓN':              region,
                'ACTIVIDAD_ECONOMICA': actividad,
                'EMPLEO':              empleo
            })
        except Exception as e:
            print(f"    [!] Error parseando: '{fecha_raw}' → {e}")

    return registros

# ── Runner principal ───────────────────────────────────────────────────────────
async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        context = await browser.new_context(viewport={"width": 1440, "height": 900})
        page = await context.new_page()

        print("Cargando Power BI...")
        await page.goto(EMBED_URL, wait_until="networkidle", timeout=90000)
        await asyncio.sleep(10)
        await page.mouse.wheel(delta_x=0, delta_y=400)
        await asyncio.sleep(3)
        await page.locator("text=EVOLUCIÓN").first.click()
        await asyncio.sleep(8)
        print("✓ Power BI cargado\n")

        todos = []
        total_regiones    = len(REGIONES_SELECCIONADAS)
        total_actividades = len(ACTIVIDADES)

        for ri, region in enumerate(REGIONES_SELECCIONADAS, 1):
            print(f"{'═'*50}")
            print(f"  REGIÓN [{ri}/{total_regiones}]: {region}")
            print(f"{'═'*50}")

            for ai, actividad in enumerate(ACTIVIDADES, 1):
                print(f"  [{ai}/{total_actividades}] {actividad}...")
                try:
                    registros = await procesar_combinacion(page, region, actividad)
                    todos.extend(registros)
                    print(f"    → {len(registros)} registros | total acumulado: {len(todos)}\n")
                except Exception as e:
                    print(f"    [✗] Error en {region} + {actividad}: {e}\n")

        await browser.close()

    # ── Exportar ───────────────────────────────────────────────────────────────
    if todos:
        df = pd.DataFrame(todos, columns=['AÑO','MES','REGIÓN','ACTIVIDAD_ECONOMICA','EMPLEO'])
        df = df.sort_values(['REGIÓN','ACTIVIDAD_ECONOMICA','AÑO','MES']).reset_index(drop=True)

        # ── Verificación de completitud ────────────────────────────────────────
        print("\n── Verificación de completitud ──")
        for region in REGIONES_SELECCIONADAS:
            for act in ACTIVIDADES:
                sub = df[(df['REGIÓN'] == region) & (df['ACTIVIDAD_ECONOMICA'] == act)]
                if sub.empty:
                    print(f"  ✗ {region} | {act}: sin datos")
                    continue
                esperados = {(a, m) for a in sub['AÑO'].unique() for m in range(1, 13)}
                obtenidos = set(zip(sub['AÑO'], sub['MES']))
                faltantes = sorted(esperados - obtenidos)
                if faltantes:
                    print(f"  ⚠ {region} | {act}: {len(faltantes)} meses faltantes → {faltantes[:5]}{'...' if len(faltantes) > 5 else ''}")
                else:
                    print(f"  ✓ {region} | {act}: {len(sub)} registros completos")

        print(f"\n✓ Total final: {len(df)} registros")
        print(df.to_string())

        # ── Nombre dinámico del archivo ────────────────────────────────────────
        if len(REGIONES_SELECCIONADAS) == 1:
            nombre_archivo = f"{REGIONES_SELECCIONADAS[0].replace(' ', '_')}_empleo.xlsx"
        elif len(REGIONES_SELECCIONADAS) <= 3:
            nombre_archivo = f"{'_'.join(r.replace(' ', '_') for r in REGIONES_SELECCIONADAS)}_empleo.xlsx"
        else:
            nombre_archivo = f"{len(REGIONES_SELECCIONADAS)}_regiones_empleo.xlsx"

        df.to_excel(nombre_archivo, index=False)
        files.download(nombre_archivo)
        print(f"\n✓ Archivo descargado: {nombre_archivo}")
    else:
        print("✗ No se obtuvieron datos")

await run()